# One common CV benchmark for model comparison

The primary comparison now uses the same saved grouped five-fold splits, repeated with seeds **20261126, 20261127 and 20261128**, for every model. The current hybrid scores **0.910474 mean CV sample-wise F1**, with repeat standard deviation **0.007372**. Its previously measured validation F1 remains **0.912393**.

This benchmark runs no additional validation evaluation and makes no model promotion. Historical CV values are preserved but are not substitutes for this common comparison score. The Kaggle score is unmeasured.

In [1]:
from pathlib import Path
import json,pandas as pd,numpy as np
root=Path.cwd();bench=root/'experiments/benchmark_v1'
summary=json.loads((bench/'summary.json').read_text())
table=pd.read_csv(bench/'comparison.csv')
table[['candidate','cv_mean','repeat_std','paired_cv_difference','repeats_improved','validation_f1']]

,candidate,cv_mean,repeat_std,paired_cv_difference,repeats_improved,validation_f1
0,Current best hybrid,0.910474,0.007372,0.000000,0,0.912393
1,Previous cluster-guided SVM,0.907289,0.008725,-0.003185,1,0.910256
2,Label-frequency correction,0.917905,0.014119,0.007431,2,0.871795
3,Graph-regularized SVM,0.908705,0.009652,-0.001769,1,0.880342
4,Other subgroups,0.914367,0.003044,0.003892,1,0.897436
5,Structured-label SVM,0.915074,0.012501,0.004600,1,0.910256
6,Unlabeled-subsample SVM ensemble,0.907289,0.017103,-0.003185,2,0.910256
7,Low-density SVM,0.910828,0.018264,0.000354,2,0.910256
8,Cluster SVM at threshold 0.60,0.907289,0.011279,-0.003185,2,0.901709
9,Year-weighted SVM,0.912243,0.010158,0.001769,1,0.905983


## What is fixed

The protocol stores the actual fold memberships and trial IDs, not only random seeds. A SHA-256 hash identifies the training CSV and another identifies the complete fold assignments. Identical embeddings, rounded to ten decimal places, form one group; multilabel stratification balances the groups across folds. Each gold row is held out exactly once per repeat. Matching originally masked duplicates are removed from the fold's training pool.

Recipes and final decoding rules are frozen before benchmarking. Each fitted transform sees only the outer training gold and permitted unlabeled training examples. Inner calibration, feature selection or parameter tuning, when part of a recipe, must remain inside that outer training fold. The runner passes query features without label columns and does not select a threshold from outer-fold outcomes.

For each repeat, the metric is sample-wise F1 over the complete out-of-fold prediction matrix, giving equal weight to each trial. The headline is the mean of the three repeat scores. The repeat standard deviation describes split sensitivity; it is not a confidence interval. Macro F1, per-reason F1, every fold's score, raw scores and decoded OOF predictions are also saved.

All models use the same outer folds, so their repeat-by-repeat differences are paired. A positive mean difference with improvement in at least two of three repeats is a descriptive shortlist criterion, not a significance test or an automatic replacement rule.

In [2]:
rows=[]
for r in summary['models']:
    result=json.loads((bench/(r['result_key']+'.json')).read_text())
    for seed,score in zip(summary['seeds'],result['repeat_scores']):
        rows.append(dict(candidate=r['candidate'],seed=seed,sample_f1=score['sample_f1'],macro_f1=score['macro_f1']))
pd.DataFrame(rows).pivot(index='candidate',columns='seed',values='sample_f1')

seed,20261126,20261127,20261128
candidate,,,
Classical tri-training,0.900212,0.915074,0.921444
Cluster SVM at threshold 0.60,0.906582,0.893843,0.921444
Current best hybrid,0.901274,0.910828,0.919321
Drug-target augmented SVM,0.912951,0.891720,0.932059
Earlier low-density SVM,0.912951,0.887473,0.932059
Embedding-kernel SVM,0.912951,0.887473,0.932059
Graph-regularized SVM,0.898089,0.906582,0.921444
Kernel-committee reason hybrid,0.910828,0.893843,0.915074
Label-frequency correction,0.925690,0.898089,0.929936


## Validation and honest interpretation

The comparison table copies already measured validation scores from the frozen recipe records. These recipes have not changed, so this benchmark does not refit or rescore validation. Incoming recipes should first pass through the same CV comparison; shortlist them, freeze them, and then evaluate validation in a small recorded batch.

Agreement between CV and validation is useful evidence, but these are development results. Earlier work used the training labels and repeatedly consulted validation. New split seeds do not turn those labels into independent test data. Continued search can also overfit the fixed CV folds. The later-year Kaggle population can differ from both the labeled training population and validation, so any chronological stress test should be reported separately and should not replace the primary CV metric.

The retained hybrid keeps its 0.912393 validation score. There is no final all-label refit and no transductive fitting on validation/test inputs in this main benchmark. The separately authorized pooled CV in notebook 11 uses original validation rows only within the training portion of a pooled fold; its results remain a distinct supplementary measurement.

## Adding a candidate

Save a JSON object with `name`, `config`, and `rule` for an implemented fold-local recipe, then run:

```sh
.venv/bin/python benchmark_cv.py --register candidate.json
```

An optional `validation` object can reference an already recorded evaluation. Otherwise its validation entry stays empty. New methods first need an adapter compatible with the experiment engine's training-position, query-feature and excluded-group interface; registration alone cannot implement an arbitrary external model.

Running `.venv/bin/python benchmark_cv.py` regenerates the comparison table from cached results where the protocol, recipe, rule and source hashes match. `--verify-only` checks the saved folds, training hash, group separation and duplicate-exclusion counts. Changing a recipe requires a new versioned name. Keep the protocol fixed across searches.

Source is embedded in notebook 04. `AGENTS.md` records this convention for future work in the project. For the methodological background, see [scikit-learn's cross-validation guidance](https://scikit-learn.org/stable/modules/cross_validation.html).

In [3]:
protocol=json.loads((bench/'protocol.json').read_text())
{k:protocol[k] for k in ['version','training_sha256','folds_sha256','seeds','folds_per_repeat','metric','limitation']}

{'version': 'part1-common-cv-v1',
 'training_sha256': 'f6019e7c0940434e671236554342c1b24983ca81f4a0ae06411a788f36c29170',
 'folds_sha256': 'b7e35fc4a6abd2351b8f97228d2253860932fd8b1f7b7432f8319b5702b7e36c',
 'seeds': [20261126, 20261127, 20261128],
 'folds_per_repeat': 5,
 'metric': 'Mean of three complete-OOF sample-wise F1 scores; equal weight per gold trial within each repeat.',
 'limitation': 'Historical tuning used these training labels and repeatedly consulted validation. This is a common development benchmark, not an independent test. Fixed folds can also be overfit through repeated search.'}

In [4]:
assert summary['validation_evaluations_performed']==0
assert summary['model_promotions_performed']==0
print('Common comparison complete; existing validation metrics reused; current submission retained.')

Common comparison complete; existing validation metrics reused; current submission retained.
